In [ ]:
import random

TARGET_VALUE = 5049
NUMBERS = "0123456789"
# POPULATION_SIZE = 100
MUTATION_RATE = 0.2
GENERATIONS = 10000


DIVERSITY_THRESHOLD = 0.8


def evaluate(expression):
    """Įvertina matematinę išraišką ir grąžina rezultatą."""
    try:
        return eval(expression)
    except (SyntaxError, ZeroDivisionError):
        return float("inf")  # Blogi išraiškos formatai atmesti


def fitness(chromosome):
    """Apskaičiuoja tinkamumo (fitness) vertę."""
    expr = NUMBERS[0]
    for i, op in enumerate(chromosome):
        expr += op + NUMBERS[i + 1]

    return abs(TARGET_VALUE - evaluate(expr))


def generate_individual():
    """Sukuria atsitiktinį individą."""
    return [random.choice(["+", "*", "-", "/"]) for _ in range(len(NUMBERS) - 1)]


def crossover(parent1, parent2):
    """Vieno taško kryžminimas."""
    point = random.randint(1, len(parent1) - 1)
    return parent1[:point] + parent2[point:]


def mutate(chromosome, fitness_value):
    """Mutacija - atsitiktinai pakeičia ženklą."""
    # Kuo mažesnis fitness, tuo geriau
    mutation_rate = MUTATION_RATE * (fitness_value / TARGET_VALUE)
    if random.random() < mutation_rate:
        index = random.randint(0, len(chromosome) - 1)
        chromosome[index] = random.choice(["+", "*", "-", "/"])
    return chromosome


def calculate_diversity(population):
    """Apskaičiuoja populiacijos įvairovę."""
    unique_individuals = len(set(tuple(ind) for ind in population))
    return unique_individuals / len(population)


def run_experiment(population_size):
    """Vykdo genetinį algoritmą ir grąžina kartų skaičių į konvergenciją ir sprendimą."""
    print(f"Starting experiment with population size: {population_size}")
    population = [generate_individual() for _ in range(population_size)]

    for generation in range(GENERATIONS):
        print(f"Generation: {generation + 1}")
        population.sort(key=fitness)

        best_fitness = fitness(population[0])
        print(f"Best fitness in generation {generation + 1}: {best_fitness}")
        best_expression = NUMBERS[0] + "".join([op + NUMBERS[i + 1] for i, op in enumerate(population[0])])
        print(f"Best Expression: {best_expression} | Evaluated: {evaluate(best_expression)}")

        if evaluate(best_expression) == TARGET_VALUE:
            print(f"Solution found in generation {generation + 1} : {best_expression}")
            return generation + 1, best_expression
        if evaluate(best_expression) != TARGET_VALUE:
            for index, individual in enumerate(population[1:]):
                expression = NUMBERS[0] + "".join([op + NUMBERS[i + 1] for i, op in enumerate(individual)])
                if evaluate(expression) == TARGET_VALUE:
                    print(
                        f"Solution found in generation {generation + 1} at index {index + 1}: {expression}"
                    )
                    return generation + 1, expression

        next_gen = [population[0]]


        while len(next_gen) < population_size:
            parent1, parent2 = random.sample(population[: population_size // 2], 2)
            child = crossover(parent1, parent2)
            child_fitness = fitness(child)
            child = mutate(child, child_fitness)
            next_gen.append(child)

        diversity = calculate_diversity(next_gen)
        if diversity < DIVERSITY_THRESHOLD:
            print(f"Diversity too low ({diversity:.2f}), repopulating...")
            num_random_individuals = int((1 - diversity) * population_size)
            next_gen.extend(
                [generate_individual() for _ in range(num_random_individuals)]
            )

        population = next_gen

    print("Solution not found within the given number of generations.")
    return generation + 1, ""



population_sizes = range(100, 210, 10)
num_repetitions = 100


results = {}
for size in population_sizes:
    results[size] = []
    for _ in range(num_repetitions):
        generations, _ = run_experiment(size)
        results[size].append(generations)

for size, generation_counts in results.items():
    avg_generations = sum(generation_counts) / len(generation_counts)
    print(
        f"Population size: {size}, Average generations to converge: {avg_generations:.2f}"
    )